In [60]:

import pandas as pd
import networkx as nx
from scipy.spatial import distance_matrix
import numpy as np
import pandas as pd
import numpy as np
from scipy.spatial import distance_matrix
from collections import deque

def load_swc(file_path):
    """
    加载SWC文件并返回DataFrame
    """
    data = []
    with open(file_path, 'r', encoding='ISO-8859-1') as f:
        for line in f:
            if line.startswith('#') or line.strip() == '':
                continue
            parts = line.strip().split()
            data.append([int(parts[0]), int(parts[1]), float(parts[2]), float(parts[3]), float(parts[4]), 
                         float(parts[5]), int(parts[6])])
    df = pd.DataFrame(data, columns=['id', 'type', 'x', 'y', 'z', 'radius', 'parent'])
    return df

def get_second_largest_component(df):
    """
    获取第二大连通块
    """
    G = nx.Graph()
    
    # 创建图，将每个节点和其父节点之间添加边
    for _, row in df.iterrows():
        if row['parent'] != -1:  # 跳过无父节点的行
            G.add_edge(row['id'], row['parent'])
    
    # 获取所有连通块
    components = list(nx.connected_components(G))
    
    # 按连通块的大小排序
    components_sorted = sorted(components, key=len, reverse=True)
    
    # 获取第二大连通块（如果存在的话）
    if len(components_sorted) > 1:
        second_largest_component = components_sorted[1]
    else:
        second_largest_component = None  # 如果没有第二大连通块，则返回None
    return df[df['id'].isin(second_largest_component)]

def calculate_closest_points(main_df, sub_tree_df):
    """
    计算主树和子树空间最接近的点
    """
    # 获取主树和子树的坐标
    main_coords = main_df[['x', 'y', 'z']].to_numpy()
    sub_tree_coords = sub_tree_df[['x', 'y', 'z']].to_numpy()
    
    # 计算主树和子树所有节点之间的欧几里得距离
    dist_matrix = distance_matrix(sub_tree_coords, main_coords)
    
    # 找到最小的距离
    min_dist_index = np.unravel_index(np.argmin(dist_matrix), dist_matrix.shape)
    sub_tree_node_id = sub_tree_df.iloc[min_dist_index[0]]['id']
    main_tree_node_id = main_df.iloc[min_dist_index[1]]['id']
    min_distance = dist_matrix[min_dist_index]
    
    print(f"最小距离的节点对：子树节点 {sub_tree_node_id} 与主树节点 {main_tree_node_id}，距离 = {min_distance:.4f}")
    
    return sub_tree_node_id, main_tree_node_id

def save_final_swc(merged_df, output_file):
    """
    保存合并后的SWC文件
    """
    merged_df.to_csv(output_file, sep=' ', header=False, index=False)
    print(f"最终处理后的 SWC 文件已保存至：{output_file}")

def generate_sub(crop_file,sort_file,output_file):

    main_df = load_swc(sort_file)
    swc_df = load_swc(crop_file)
    sub_tree_df=get_second_largest_component(swc_df)
    save_final_swc(sub_tree_df,output_file)
    sub_tree_node_id, main_tree_node_id=calculate_closest_points(main_df,sub_tree_df)
    soma_node=int(sub_tree_node_id)
    main_node=int(main_tree_node_id)
    return soma_node,main_node

import pandas as pd
import numpy as np
from scipy.spatial import distance_matrix
from collections import deque
from tqdm import tqdm
def read_swc(filename):
    nodes = {}
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            if len(parts) != 7:
                print(f"警告：行格式不正确，已忽略：{line}")
                continue
            n = int(parts[0])
            T = int(parts[1])
            x = float(parts[2])
            y = float(parts[3])
            z = float(parts[4])
            radius = float(parts[5])
            P = int(float(parts[6]))
            nodes[n] = {'n': n, 'T': T, 'x': x, 'y': y, 'z': z, 'radius': radius, 'P': P, 'children': []}

    for node in nodes.values():
        P = node['P']
        if P != -1 and P in nodes:
            nodes[P]['children'].append(node['n'])
        elif P != -1:
            print(f"警告：父节点{P}不存在，节点{node['n']}的父节点设为-1")
            node['P'] = -1
    return nodes

def save_to_swc(df, filename):
    with open(filename, 'w') as f:
        f.write("# 标准化后的SWC文件\n")
        for idx, row in df.iterrows():
            n = int(row['n'])
            T = int(row['T'])
            x = float(row['x'])
            y = float(row['y'])
            z = float(row['z'])
            radius = float(row['radius'])
            P = int(row['P'])
            f.write(f"{n} {T} {x} {y} {z} {radius} {P}\n")
    print(f"已保存为 {filename}")

def assign_new_ids_to_subtree(sub_tree_df, max_main_id):
    updated_sub_tree_df = sub_tree_df.copy()
    id_mapping = {}
    for idx, row in updated_sub_tree_df.iterrows():
        new_id = max_main_id + 1
        updated_sub_tree_df.at[idx, 'n'] = new_id
        id_mapping[row['n']] = new_id
        max_main_id = new_id
    updated_sub_tree_df['P'] = updated_sub_tree_df['P'].map(id_mapping).fillna(updated_sub_tree_df['P'])
    return updated_sub_tree_df, id_mapping

def update_soma_nodes(updated_sub_tree_df, main_tree_node_id):
    soma_nodes = updated_sub_tree_df[updated_sub_tree_df['P'] == -1]
    for idx, row in soma_nodes.iterrows():
        soma_node_id = row['n']
        updated_sub_tree_df.at[idx, 'P'] = main_tree_node_id
        updated_sub_tree_df.at[idx, 'T'] = 0
        print(f"胞体节点 {soma_node_id} 的父节点ID已更新为主树节点 {main_tree_node_id}，类型已更新为0")
    return updated_sub_tree_df

# def find_path(nodes, start_node, end_node):
#     queue = deque([(start_node, [start_node])])
#     visited = {start_node}
#     while queue:
#         current_node, path = queue.popleft()
#         if current_node == end_node:
#             return path
#         for neighbor in nodes.get(current_node, {}).get('children', []):
#             if neighbor not in visited:
#                 visited.add(neighbor)
#                 queue.append((neighbor, path + [neighbor]))
#     return None


# def reverse_path(nodes, path):
#     path_len = len(path)
#     for i in range(path_len - 1):
#         child_id = path[i]
#         parent_id = path[i + 1]
#         nodes[child_id]['P'] = parent_id
#     for node_id in path:
#         nodes[node_id]['children'] = []
#     for i in range(path_len - 1):
#         parent_id = path[i + 1]
#         child_id = path[i]
#         nodes[parent_id]['children'].append(child_id)

# def adjust_soma_and_roots(nodes, soma_node):
#     if isinstance(nodes, dict):
#         nodes = pd.DataFrame.from_dict(nodes, orient='index')
    
#     original_soma_nodes = [node['n'] for _, node in nodes.iterrows() if node['P'] == -1 and node['n'] != soma_node]
#     for old_soma_node in original_soma_nodes:
#         path = find_path(nodes, old_soma_node, soma_node)
        
#         if path:
#             reverse_path(nodes, path)
#         else:
#             print(f"无法找到从旧胞体节点 {old_soma_node} 到新胞体节点 {soma_node} 的路径")
#     #print("hh")
#     nodes.loc[nodes['n'] == soma_node, 'T'] = 1
#     nodes.loc[nodes['n'] == soma_node, 'P'] = -1
#     nodes.loc[nodes['n'] != soma_node, 'T'] = 0
#     nodes['children'] = None
#     for idx, row in nodes.iterrows():
#         parent_id = row['P']
#         if parent_id != -1:
#             print(parent_id)
#             parent_node = nodes[nodes['n'] == parent_id]
#             if not parent_node.empty:
#                 # print(parent_node)
#                 parent_node_idx = parent_node.index[0]
#                 if pd.isnull(nodes.at[parent_node_idx, 'children']):
#                     nodes.at[parent_node_idx, 'children'] = []
#                 nodes.at[parent_node_idx, 'children'].append(row['n'])
# \
#     return nodes


def find_path(nodes, start_node, end_node):
    queue = deque([(start_node, [start_node])])
    visited = {start_node}
    while queue:
        current_node, path = queue.popleft()
        if current_node == end_node:
            return path
        for neighbor in nodes.get(current_node, {}).get('children', []):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, path + [neighbor]))
    return None


def reverse_path(nodes, path):
    path_len = len(path)
    for i in range(path_len - 1):
        child_id = path[i]
        parent_id = path[i + 1]
        nodes[child_id]['P'] = parent_id
    for node_id in path:
        nodes[node_id]['children'] = []
    for i in range(path_len - 1):
        parent_id = path[i + 1]
        child_id = path[i]
        nodes[parent_id]['children'].append(child_id)


def adjust_soma_and_roots(nodes,soma_node):
    # 使用加权后的胞体检测方法
    # soma_node = find_potential_soma_with_adjusted_weights(nodes)
    # print(f"选择节点 {soma_node} 作为新的胞体节点")

    # 找到原始的 P=-1 节点（根节点）
    original_soma_nodes = [node['n'] for node in nodes.values() if node['P'] == -1 and node['n'] != soma_node]
    #print(f"原始 P=-1 节点: {original_soma_nodes}")

    # 处理每个原始根节点到新胞体节点的路径
    for old_soma_node in original_soma_nodes:
        path = find_path(nodes, old_soma_node, soma_node)
        if path:
            #print(f"从旧胞体节点 {old_soma_node} 到新胞体节点 {soma_node} 的路径: {path}")
            reverse_path(nodes, path)
        else:
            print(f"无法找到从旧胞体节点 {old_soma_node} 到新胞体节点 {soma_node} 的路径")

    # 设置新胞体节点的类型和父节点
    nodes[soma_node]['T'] = 1
    nodes[soma_node]['P'] = -1
    for node in nodes.values():
        if node['n'] != soma_node:
            node['T'] = 0  # 非胞体节点的类型设为1

    # 重建父子关系
    for node in nodes.values():
        node['children'] = []
    for node in nodes.values():
        P = node['P']
        if P != -1 and P in nodes:
            nodes[P]['children'].append(node['n'])
        elif P != -1:
            print(f"警告：父节点 {P} 不存在，节点 {node['n']} 的父节点设为 -1")
            node['P'] = -1
    return nodes

def write_swc(nodes, filename):
    with open(filename, 'w') as f:
        f.write("# 标准化后的SWC文件\n")
        for n in sorted(nodes.keys()):
            node = nodes[n]
            f.write(f"{node['n']} {node['T']} {node['x']} {node['y']} {node['z']} {node['radius']} {node['P']}\n")

# sub_tree_df= read_swc( r"D:\GXQ\cross_em_lm\test_data\reconnect\39490508916_sub.swc")
# nodes=adjust_soma_and_roots(sub_tree_df,18138)
# write_swc(nodes,r"D:\GXQ\cross_em_lm\test_data\reconnect\39490508916_sub123.swc")
def process_swc_files(main_file, sub_file, output_file, soma_node,main_node):
    sub_tree_df = read_swc(sub_file)
    main_df = read_swc(main_file)
    main_max_id = len(main_df)
    
    main_df = pd.DataFrame.from_dict(main_df, orient='index')
    main_df.reset_index(drop=True, inplace=True)
    
    nodes = adjust_soma_and_roots(sub_tree_df, soma_node)
    #nodes=adjust_soma_and_roots(sub_tree_df,18138)
    write_swc(nodes,sub_file)
    nodes = pd.DataFrame.from_dict(nodes, orient='index')
    nodes.reset_index(drop=True, inplace=True)
    

    updated_sub_tree_df, id_mapping = assign_new_ids_to_subtree(nodes, main_max_id)
    #print(updated_sub_tree_df)
    updated_sub_tree_df = update_soma_nodes(updated_sub_tree_df, main_node)
    
    hh_main_df = pd.concat([main_df, updated_sub_tree_df], ignore_index=True)
    save_to_swc(hh_main_df, output_file)



# # # 示例调用
# if __name__ == "__main__":

#     sort_file = r"D:\GXQ\cross_em_lm\test_data\reconnect\50799931696_sort.swc"
# # # #main_df = read_swc(r"D:\GXQ\cross_em_lm\test_data\reconnect\50799931696_sort.swc")
#     swc_file = r"D:\GXQ\cross_em_lm\test_data\reconnect\50799931696.swc"
#     sub_file = r"D:\GXQ\cross_em_lm\test_data\reconnect\50799931696_sub.swc"
#     final_file = r"D:\GXQ\cross_em_lm\test_data\reconnect\50799931696_reconnect.swc"
# # # # 加载 SWC 数据
#     soma_node=generate_sub(swc_file,sort_file,sub_file)
#     process_swc_files(sort_file, sub_file, final_file, soma_node)
import pandas as pd
import os
if __name__ == "__main__":
    crop_dir=r"Z:\SEU-ALLEN\Users\XiaoqinGu\dataset\H01\v1\crop_original"
    sort_dir=r"Z:\SEU-ALLEN\Users\XiaoqinGu\dataset\H01\v2\lcc_sort"
    sub_dir=r"Z:\SEU-ALLEN\Users\XiaoqinGu\dataset\H01\v2\level3\sub"
    final_dir=r"Z:\SEU-ALLEN\Users\XiaoqinGu\dataset\H01\v2\level3\final"
    csv=pd.read_csv(r"Z:\SEU-ALLEN\Users\XiaoqinGu\dataset\H01\data_below80.csv")
    filtered_file = csv[csv['LCC_Node_Proportion'] >= 70]['Filename']
    #print(filtered_file)
    for f in filtered_file[2:]:
    #     if f == "39490508916.swc":
    # for f in tqdm(filtered_file, desc="Processing files", unit="file"):
        print(f)
        crop_swc=os.path.join(crop_dir,f)
        sort_swc=os.path.join(sort_dir,str(f).replace(".swc","_sort.swc"))
        sub_swc=os.path.join(sub_dir,str(f).replace(".swc","_sub.swc"))
        final_swc=os.path.join(final_dir,str(f).replace(".swc","_reconnect.swc"))
        soma_node,main_node=generate_sub(crop_swc,sort_swc,sub_swc)
        process_swc_files(sort_swc, sub_swc, final_swc, soma_node, main_node)




57017008473.swc
最终处理后的 SWC 文件已保存至：Z:\SEU-ALLEN\Users\XiaoqinGu\dataset\H01\v2\level3\sub\57017008473_sub.swc
最小距离的节点对：子树节点 42822.0 与主树节点 36085.0，距离 = 248.5739
胞体节点 37211 的父节点ID已更新为主树节点 36085，类型已更新为0
已保存为 Z:\SEU-ALLEN\Users\XiaoqinGu\dataset\H01\v2\level3\final\57017008473_reconnect.swc
3093748739.swc
最终处理后的 SWC 文件已保存至：Z:\SEU-ALLEN\Users\XiaoqinGu\dataset\H01\v2\level3\sub\3093748739_sub.swc
最小距离的节点对：子树节点 13524.0 与主树节点 10202.0，距离 = 1876.5716
胞体节点 13523 的父节点ID已更新为主树节点 10202，类型已更新为0
已保存为 Z:\SEU-ALLEN\Users\XiaoqinGu\dataset\H01\v2\level3\final\3093748739_reconnect.swc
40404502148.swc
最终处理后的 SWC 文件已保存至：Z:\SEU-ALLEN\Users\XiaoqinGu\dataset\H01\v2\level3\sub\40404502148_sub.swc
最小距离的节点对：子树节点 5030.0 与主树节点 108.0，距离 = 345.0058
胞体节点 5030 的父节点ID已更新为主树节点 108，类型已更新为0
已保存为 Z:\SEU-ALLEN\Users\XiaoqinGu\dataset\H01\v2\level3\final\40404502148_reconnect.swc
59882147771.swc
最终处理后的 SWC 文件已保存至：Z:\SEU-ALLEN\Users\XiaoqinGu\dataset\H01\v2\level3\sub\59882147771_sub.swc
最小距离的节点对：子树节点 12626.0 与主树节点 7200.0，距离 

In [ ]:
# import pandas as pd
# import os
# if __name__ == "__main__":
#     crop_dir=r"Z:\SEU-ALLEN\Users\XiaoqinGu\dataset\H01\v1\crop_original"
#     sort_dir=r"Z:\SEU-ALLEN\Users\XiaoqinGu\dataset\H01\v2\lcc_sort"
#     sub_dir=r"Z:\SEU-ALLEN\Users\XiaoqinGu\dataset\H01\v2\level3\sub"
#     sub_dir=r"Z:\SEU-ALLEN\Users\XiaoqinGu\dataset\H01\v2\level3\final"
#     csv=pd.read_csv(r"Z:\SEU-ALLEN\Users\XiaoqinGu\dataset\H01\data_below80.csv")
#     filtered_file = csv[csv['LCC_Node_Proportion'] >= 70]['Filename']
#     for f in filtered_file:
#         crop_swc=os.path.join(crop_dir,f)
#         sort_swc=os.path.join(sort_dir,str(f).replace(".swc","_sort.swc"))
#         sub_swc=os.path.join(sub_dir,str(f).replace(".swc","_sub.swc"))
#         final_swc=os.path.join(final_dir,str(f).replace(".swc","_reconnect.swc"))
#         soma_node=generate_sub(crop_swc,sort_swc,sub_swc)
#         soma_node=generate_sub(swc_file,sort_file,sub_file)
#         process_swc_files(sort_file, sub_file, final_file, soma_node)
#     #print(main_swc,sort_swc)